In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
# import xgboost as xgb
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.logei import qLogNoisyExpectedImprovement
import seaborn

In [2]:
o = 27
n = 23
s = o-n
sampler = Sampler_class()
Parameters_lis = [
    RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0,1])),
    RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0,1])),
    ]

In [3]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/Modelling/ModelMk4.json")
gp_model.get_next_trials(max_trials=1)
def SurrogateModelOfReality(s1, s2, b1):
    y_pred = gp_model.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
    return np.float64(y_pred)

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


In [4]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(
            name="s1", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="s2", parameter_type="float", bounds=(0, 1)
        ),
        RangeParameterConfig(
            name="b1", parameter_type="float", bounds=(0, 1)
        ),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": qLogNoisyExpectedImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    X = sampler.three.PseudorandomSampler3D_func(n,Parameters_lis).T

    for array in X:
        my_parameters = {"s1": array[0], "s2": array[1], "b1": array[2]}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(**my_parameters)})

    for _ in range(s): # Run 10 rounds of trials
        # We will request three trials at a time in this example
        trials = client.get_next_trials(max_trials=1)

        for trial_index, parameters in trials.items():
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]

            result = SurrogateModelOfReality(s1, s2, b1)

            # Set raw_data as a dictionary with metric names as keys and results as values
            raw_data = {metric_name: result}

            # Complete the trial with the result
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)
    # print(client.summarize())
    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(client.summarize().t1))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
14.378943096147458

Trial 1 =========================================
14.363898996405654

Trial 2 =========================================
14.101503952498613

Trial 3 =========================================
14.985700400570435

Trial 4 =========================================
14.3719950537623

Trial 5 =========================================
14.027779068689826

Trial 6 =========================================
14.344344889166623

Trial 7 =========================================
16.314988671917504

Trial 8 =========================================
16.270791785952973

Trial 9 =========================================
15.951633469912856

Trial 10 =========================================
13.821357198157816

Trial 11 =========================================
13.695599014851073

Trial 12 =========================================
13.993494558016627

Trial 13 =========================================
14.216107591786578

Trial 14 =========

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/gpytorch/distributions/multivariate_normal.py:376: NumericalWarning: Negative variance values detected. This is likely due to numerical instabilities. Rounding negative variances up to 1e-10.
  warnings.warn(


Trial 16 =========================================
13.855483433512926

Trial 17 =========================================
14.24698170354894

Trial 18 =========================================
13.384725794363774

Trial 19 =========================================
15.533942467350526

Trial 20 =========================================
13.408050474406046

Trial 21 =========================================
14.017209450625348

Trial 22 =========================================
13.863343245700037

Trial 23 =========================================
14.136790391597003

Trial 24 =========================================
13.806862320995513

Trial 25 =========================================
14.102804111130336



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 26 =========================================
13.083918975040056

Trial 27 =========================================
16.397202825416006

Trial 28 =========================================
13.87761213355056



...

Trial 74 =========================================
15.672574486771369

Trial 75 =========================================
13.435804725192332

Trial 76 =========================================
16.350993613481005

Trial 77 =========================================
16.19056588157404

Trial 78 =========================================
14.634996608710246

Trial 79 =========================================
15.369445612829756

Trial 80 =========================================
15.223914428745424

Trial 81 =========================================
14.10175762251913

Trial 82 =========================================
16.00803909710072

Trial 83 =========================================
15.885476765707244

Trial 84 =========================================
13.56424722972997

Trial 85 =========================================
13.70461807138775

Trial 86 =========================================
13.805189123934397

Trial 87 =========================================
15.801250289501569

Trial 88 ==

/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 91 =========================================
16.210082181098365

Trial 92 =========================================
15.052439135284548



/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(
/Users/thomasdodd/miniconda3/envs/Ax1_env/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:40: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


Trial 93 =========================================
15.633066488457928

Trial 94 =========================================
15.703650572710142

Trial 95 =========================================
13.910340620012603

Trial 96 =========================================
15.023040573634765

Trial 97 =========================================
16.50748792383692

Trial 98 =========================================
16.45962748976818

Trial 99 =========================================
14.10454723053494

[14.3789431  14.363899   14.10150395 14.9857004  14.37199505 14.02777907
 14.34434489 16.31498867 16.27079179 15.95163347 13.8213572  13.69559901
 13.99349456 14.21610759 16.07659734 16.49471244 13.85548343 14.2469817
 13.38472579 15.53394247 13.40805047 14.01720945 13.86334325 14.13679039
 13.80686232 14.10280411 13.08391898 16.39720283 13.87761213 15.53394247
 14.21562824 15.09521304 15.27996785 17.57874189 14.41196184 13.99040989
 14.84165853 16.14058102 15.36684858 16.12326888 14.90186461 15.53394

In [5]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 17.57874189431595
Avg = 14.790700249573938
Std = 1.0345265727593749


In [6]:
print(y_max_arr.tolist())

[14.378943096147458, 14.363898996405654, 14.101503952498613, 14.985700400570435, 14.3719950537623, 14.027779068689826, 14.344344889166623, 16.314988671917504, 16.270791785952973, 15.951633469912856, 13.821357198157816, 13.695599014851073, 13.993494558016627, 14.216107591786578, 16.076597342078248, 16.49471244234516, 13.855483433512926, 14.24698170354894, 13.384725794363774, 15.533942467350526, 13.408050474406046, 14.017209450625348, 13.863343245700037, 14.136790391597003, 13.806862320995513, 14.102804111130336, 13.083918975040056, 16.397202825416006, 13.87761213355056, 15.533942467350526, 14.215628243875692, 15.095213037609497, 15.279967845693628, 17.57874189431595, 14.411961841415103, 13.990409886565377, 14.841658529235186, 16.140581022705824, 15.366848575136299, 16.123268883732305, 14.90186460561392, 15.533942467350526, 13.930966503427411, 13.535850945721865, 14.408621021580034, 14.912330851993863, 13.155478509894486, 14.392019583621758, 16.04053512363729, 14.738184206519403, 15.0443

In [7]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/Sandbox/SequentialTestingOfSamplingTechnique/DataGenerated/BOpt-EI-9,27,1-23.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)